# Merge Linguistic Proximity (prox1) Country-Pair Lookup

Reads the raw Melitz & Toubal (2014) `prox1` table
(`data/gravity/melitz_toubal_proxling.dta` -- the same source Bekes and Ottaviano 2025
use for their own language-similarity variable) and resolves it into a complete,
closed country-pair lookup over every ISO3 code that appears anywhere in our match
data: self-pairs set to 1.0, Monaco (absent from the raw table) resolved via the
same fallback rule used everywhere else in this project (Monaco vs. a
French-official country -> 1.0, Monaco vs. anyone else -> France's value as a
stand-in), and any other missing pair (mostly involving South Korea, entirely
absent from this table's country coverage) falls back to 0.

**Output**: `data/gravity/ling_prox_pairs_final.csv` -- a complete lookup requiring zero
further fallback logic downstream. `homophily.ipynb` reads this file directly for
Section 6/6.1/6.2's random-matching benchmark and field-composition calculations
(which query arbitrary same-tournament country pairs, not just pairs realized as
actual matches, so this can't be pre-baked into the match-level panel the way
`winners/losers_ling_prox_cont` already is upstream in `final_ds.ipynb`).

**Pipeline position**: independent of `final_ds.ipynb`/`homophily.ipynb` -- run
whenever `melitz_toubal_proxling.dta` changes or the country universe in the match
data expands. Does not touch `men_matches_with_ranks_cleaned.xlsx`, `team_gs_panel.csv`,
or `tiebreak_panel.csv` -- those already carry the per-match `prox1` columns via
`final_ds.ipynb` (see the warning cell there about not re-running it from raw inputs).

In [1]:
import os
import pandas as pd
from itertools import combinations

ROOT       = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
EXCEL_PATH = os.path.join(ROOT, 'data', 'atp', 'men_matches_with_ranks_cleaned.xlsx')
PROX_PATH  = os.path.join(ROOT, 'data', 'gravity', 'melitz_toubal_proxling.dta')
OUT_PATH   = os.path.join(ROOT, 'data', 'gravity', 'ling_prox_pairs_final.csv')
print('Paths set.')

Paths set.


## 1. Country universe: every ISO3 appearing in our match data

In [2]:
iso_cols = ['winners_p1_iso3', 'winners_p2_iso3', 'losers_p1_iso3', 'losers_p2_iso3']
raw = pd.read_excel(EXCEL_PATH, sheet_name='players_list', usecols=iso_cols)

isos = set()
for c in iso_cols:
    isos |= set(raw[c].dropna().unique())
isos = sorted(isos)
print(f'Country universe: {len(isos)} ISO3 codes')

Country universe: 65 ISO3 codes


## 2. Raw prox1 lookup + Monaco/KOR fallback resolution

In [3]:
prox_raw = pd.read_stata(PROX_PATH)
raw_lut = {}
for _, r in prox_raw.iterrows():
    if pd.isna(r['iso3_o']) or pd.isna(r['iso3_d']) or pd.isna(r['proxling']):
        continue
    raw_lut[tuple(sorted([r['iso3_o'], r['iso3_d']]))] = float(r['proxling'])
print(f'Raw Melitz-Toubal country-pairs: {len(raw_lut):,}')

FRENCH_OFFICIAL = {'FRA', 'BEL', 'CHE', 'CAN', 'LUX'}

def resolve_prox1(a, b):
    """Same fallback rule used throughout this project (final_ds.ipynb, homophily.ipynb):
    MCO (absent from the raw table) vs. a French-official country -> 1.0; MCO vs.
    anyone else -> France's value as a stand-in. Any other missing pair (mostly KOR,
    entirely absent from this 1990s-2000s country coverage) falls back to 0."""
    if a == b:
        return 1.0
    if a == 'MCO' or b == 'MCO':
        other = b if a == 'MCO' else a
        if other in FRENCH_OFFICIAL:
            return 1.0
        return raw_lut.get(tuple(sorted(['FRA', other])), 0.0)
    return raw_lut.get(tuple(sorted([a, b])), 0.0)

Raw Melitz-Toubal country-pairs: 21,736


## 3. Build the complete, closed lookup table

In [4]:
rows = []
n_fallback = 0
for a, b in combinations(isos, 2):
    key = tuple(sorted([a, b]))
    in_raw_table = key in raw_lut or a == 'MCO' or b == 'MCO'
    val = resolve_prox1(a, b)
    if not in_raw_table:
        n_fallback += 1
    rows.append({'iso3_a': a, 'iso3_b': b, 'prox1': val})
for a in isos:
    rows.append({'iso3_a': a, 'iso3_b': a, 'prox1': 1.0})

pairs_final = pd.DataFrame(rows).sort_values(['iso3_a', 'iso3_b']).reset_index(drop=True)
print(f'Pairs exported: {len(pairs_final):,} ({len(isos)} countries, self-pairs included)')
print(f'Pairs resolved via fallback (not in raw table): {n_fallback}')
print(f'prox1 range: [{pairs_final["prox1"].min():.3f}, {pairs_final["prox1"].max():.3f}]')
assert pairs_final['prox1'].between(0, 1).all(), 'prox1 must be within [0,1]'
assert pairs_final['prox1'].notna().all(), 'prox1 must have no missing values'

Pairs exported: 2,145 (65 countries, self-pairs included)
Pairs resolved via fallback (not in raw table): 125
prox1 range: [0.000, 1.000]


## 4. Save

In [5]:
pairs_final.to_csv(OUT_PATH, index=False)
print(f'Saved {len(pairs_final)} rows -> {OUT_PATH}')

Saved 2145 rows -> C:\Users\aldi\Documents\GitHub\tennis-homophily\data\gravity\ling_prox_pairs_final.csv
